# How to Use Any Imputer in Scikit-Learn (`EMImputer`, `SimpleImputer`, `KNNImputer`, `IterativeImputer`)

This notebook demonstrates the standard, leak-free workflow for using **any** scikit-learn–compatible imputer:
1. **Split first** (`train_test_split`) before inspecting or fitting on data.
2. **Standalone `fit` / `transform`**: Fit strictly on `X_train` and apply learned parameters to `X_test`.
3. **Drop-in interchangeability**: Swap between `EMImputer`, `SimpleImputer`, `KNNImputer`, and `IterativeImputer` using the exact same API (`fit`, `transform`, `fit_transform`).
4. **Mixed-type `ColumnTransformer` + `Pipeline`**: Combine numeric, categorical, and boolean imputers inside a single cross-validated or end-to-end modeling pipeline.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imputer import EMImputer

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but EMImputer was fitted with feature names",
    category=UserWarning,
)

## 1. Create Sample Dataset & Split Before Any Fitting

In [ ]:
rng = np.random.default_rng(42)
n = 200

x1 = rng.normal(loc=50.0, scale=10.0, size=n)
x2 = 1.8 * x1 + rng.normal(loc=5.0, scale=4.0, size=n)
x3 = -0.7 * x1 + 0.4 * x2 + rng.normal(loc=0.0, scale=5.0, size=n)

df = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "region": rng.choice(["north", "south", "east"], size=n),
    "is_active": pd.Series(rng.choice([True, False], size=n), dtype="boolean"),
})

logits = 0.08 * (x1 - 50.0) + 0.04 * (x2 - 95.0) - 0.05 * x3
y = (rng.random(n) < 1.0 / (1.0 + np.exp(-logits))).astype(int)

# Inject ~15% missingness into numeric columns
for col in ["x1", "x2", "x3"]:
    na_idx = rng.choice(n, size=int(0.15 * n), replace=False)
    df.loc[na_idx, col] = np.nan

# Inject missing values into categorical and nullable boolean columns
df.loc[rng.choice(n, size=8, replace=False), "region"] = np.nan
df.loc[rng.choice(n, size=6, replace=False), "is_active"] = pd.NA

# Split BEFORE fitting or selecting columns
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.25, random_state=42, stratify=y
)

num_cols = X_train.select_dtypes(include="float").columns
cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns
bool_cols = X_train.select_dtypes(include=["bool", "boolean"]).columns

print("Train missing counts:", X_train[num_cols].isna().sum().to_dict())
print("Test missing counts: ", X_test[num_cols].isna().sum().to_dict())

## 2. Standalone Pattern: `fit(X_train)` -> `transform(X_test)`

Every scikit-learn imputer follows the exact same contract:
- `imputer.fit(X_train)` learns imputation statistics/parameters **only** from training data.
- `imputer.transform(X_test)` fills missing values in new/unseen data using those frozen parameters.

In [ ]:
imputers = {
    "EMImputer": EMImputer(max_iter=200, tol=1e-5, ridge=1e-6),
    "SimpleImputer (mean)": SimpleImputer(strategy="mean"),
    "SimpleImputer (median)": SimpleImputer(strategy="median"),
    "KNNImputer (k=5)": KNNImputer(n_neighbors=5),
    "IterativeImputer (MICE)": IterativeImputer(random_state=42, max_iter=20),
}

print("First 5 rows of X_test[num_cols] BEFORE imputation:")
display(X_test[num_cols].head())

for name, imp in imputers.items():
    imp.fit(X_train[num_cols])  # fit ONLY on train
    X_test_imp = imp.transform(X_test[num_cols])  # transform test
    X_test_imp_df = pd.DataFrame(X_test_imp, columns=num_cols, index=X_test.index)
    remaining_nans = int(np.isnan(X_test_imp_df.to_numpy()).sum())
    print(f"\n--- {name} (remaining NaNs: {remaining_nans}) ---")
    display(X_test_imp_df.head())

## 3. Swapping Any Imputer Inside a Mixed-Type `Pipeline`

Wrapping your imputer inside `ColumnTransformer` + `Pipeline` ensures zero data leakage during both `train_test_split` evaluation and `cross_val_score`.

In [ ]:
def build_pipeline(numeric_imputer):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_imputer, num_cols),
            (
                "cat",
                Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore")),
                ]),
                cat_cols,
            ),
            ("bool", SimpleImputer(strategy="most_frequent"), bool_cols),
        ],
        remainder="drop",
    )
    return Pipeline([
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=1000)),
    ])

results = []
for name, imp in [
    ("EMImputer", EMImputer(max_iter=200, tol=1e-5, ridge=1e-6)),
    ("SimpleImputer (mean)", SimpleImputer(strategy="mean")),
    ("SimpleImputer (median)", SimpleImputer(strategy="median")),
    ("KNNImputer (k=5)", KNNImputer(n_neighbors=5)),
    ("IterativeImputer", IterativeImputer(random_state=42, max_iter=20)),
]:
    pipe = build_pipeline(imp)
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy").mean()
    pipe.fit(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    results.append({"Imputer": name, "5-Fold CV Accuracy": round(cv_acc, 3), "Test Accuracy": round(test_acc, 3)})

display(pd.DataFrame(results))